# 05 - Démonstration interactive de BLIP et GIT

Ce notebook propose une **démonstration interactive** des modèles vision-langage légers **BLIP** (`Salesforce/blip-image-captioning-base`) et **GIT** (`microsoft/git-base`).

Il permet de :
1. Charger une image personnalisée ou utiliser une image d'exemple.
2. Choisir le modèle (**BLIP** ou **GIT**) et le niveau de quantification (**fp16**, **int8**, **int4**, ou **fp32**).
3. **Générer une légende automatique** pour l'image.
4. **Juger la cohérence d'une légende** candidate à l'aide de la perte *cross-entropy* et d'un score d'alignement.
5. **Comparer deux légendes candidates** (sélection de la description la plus pertinente).
6. Lancer une **interface Web Gradio** complète directement dans Colab avec un lien de partage public (`share=True`).

**Prérequis matériel :** Session Google Colab avec GPU T4 (12 Go de VRAM maximum).

In [ ]:
# Installer les bibliothèques requises dans Google Colab.
!pip -q install datasets transformers accelerate bitsandbytes gradio pillow matplotlib

In [ ]:
# Connecter Google Drive et configurer les chemins du projet.
from pathlib import Path
import gc
import math
import time
import torch
from PIL import Image

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ia_visuelle")
except Exception:
    PROJECT_DIR = Path("./")

RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR = PROJECT_DIR / "data"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositif matériel détecté : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU actif : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Mémoire GPU totale : {total_mem:.2f} Go")
else:
    print("Attention : Exécution sur CPU. Les quantifications int8/int4 nécessitent un GPU CUDA.")

## 1. Fonctions de chargement, quantification et inférence

Les fonctions ci-dessous permettent de charger les modèles en mémoire en gérant la libération du GPU (`clear_gpu`), la quantification avec `BitsAndBytesConfig`, ainsi que le calcul de la perte (*cross-entropy loss*) servant de score de jugement d'alignement.

In [ ]:
from transformers import (
    BlipForConditionalGeneration,
    BlipProcessor,
    GitForCausalLM,
    GitProcessor,
    BitsAndBytesConfig,
)

MODEL_CONFIGS = {
    "blip": "Salesforce/blip-image-captioning-base",
    "git": "microsoft/git-base",
}


def clear_gpu():
    """Libérer le cache PyTorch et forcer le ramasse-miettes."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_quantization_config(precision):
    """Créer la configuration bitsandbytes selon la précision demandée."""
    if precision == "int8":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "int4":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    return None


def load_model_and_processor(model_name="blip", precision="fp16"):
    """Charger BLIP ou GIT avec la quantification demandée."""
    model_name = model_name.lower().strip()
    model_id = MODEL_CONFIGS[model_name]
    is_cuda = (DEVICE == "cuda")

    q_config = get_quantization_config(precision) if is_cuda else None
    dtype = torch.float16 if (is_cuda and precision == "fp16") else torch.float32

    if model_name == "blip":
        processor = BlipProcessor.from_pretrained(model_id)
        if q_config is not None:
            model = BlipForConditionalGeneration.from_pretrained(
                model_id,
                quantization_config=q_config,
                device_map="auto",
            )
        else:
            model = BlipForConditionalGeneration.from_pretrained(
                model_id,
                torch_dtype=dtype,
                device_map="auto" if is_cuda else None,
            )
            if not is_cuda:
                model = model.to("cpu")
    else:  # git
        processor = GitProcessor.from_pretrained(model_id)
        if q_config is not None:
            model = GitForCausalLM.from_pretrained(
                model_id,
                quantization_config=q_config,
                device_map="auto",
            )
        else:
            model = GitForCausalLM.from_pretrained(
                model_id,
                torch_dtype=dtype,
                device_map="auto" if is_cuda else None,
            )
            if not is_cuda:
                model = model.to("cpu")

    model.eval()
    return model, processor


def generate_image_caption(model, processor, image, model_name="blip", max_new_tokens=50):
    """Générer une légende textuelle pour une image."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        if model_name == "git":
            out = model.generate(pixel_values=inputs.get("pixel_values"), max_new_tokens=max_new_tokens)
        else:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(out[0], skip_special_tokens=True).strip()


def compute_alignment_loss(model, processor, image, caption):
    """Calculer la perte cross-entropy entre une image et une légende candidate."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    inputs = processor(images=image, text=caption, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        result = model(**inputs, labels=inputs["input_ids"])
    return float(result.loss.detach().cpu().item())


def judge_single_caption(model, processor, image, caption):
    """Évaluer la cohérence d'une légende pour une image."""
    loss = compute_alignment_loss(model, processor, image, caption)
    score = -loss
    perplexity = math.exp(min(loss, 20.0))
    if loss <= 2.5:
        verdict = "Cohérence Forte (légende très bien alignée)"
    elif loss <= 3.8:
        verdict = "Cohérence Moyenne (légende plausible)"
    else:
        verdict = "Incohérence (légende mal alignée avec l'image)"
    return {
        "loss": round(loss, 4),
        "score": round(score, 4),
        "perplexity": round(perplexity, 2),
        "verdict": verdict,
    }


def compare_captions(model, processor, image, caption_a, caption_b):
    """Comparer deux légendes alternatives et déterminer la meilleure."""
    loss_a = compute_alignment_loss(model, processor, image, caption_a)
    loss_b = compute_alignment_loss(model, processor, image, caption_b)
    winner = "Légende A" if loss_a < loss_b else ("Légende B" if loss_b < loss_a else "Égalité")
    return {
        "loss_a": round(loss_a, 4),
        "score_a": round(-loss_a, 4),
        "loss_b": round(loss_b, 4),
        "score_b": round(-loss_b, 4),
        "winner": winner,
        "delta": round(abs(loss_a - loss_b), 4),
    }

print("Fonctions d'inférence et d'évaluation initialisées avec succès.")

## 2. Test programmatique sur une image d'exemple

Nous testons ici **BLIP en fp16** sur une image d'exemple avec :
- Une légende fidèle (bonne cohérence)
- Une légende perturbatrice (incohérente)

In [ ]:
import matplotlib.pyplot as plt
import urllib.request

# Télécharger une image d'exemple
sample_url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=600"
test_image_path = Path("sample_dog.jpg")
if not test_image_path.exists():
    try:
        urllib.request.urlretrieve(sample_url, test_image_path)
    except Exception:
        # Image synthétique en cas d'absence d'accès réseau
        img_fallback = Image.new("RGB", (300, 300), color=(120, 180, 220))
        img_fallback.save(test_image_path)

sample_img = Image.open(test_image_path).convert("RGB")

print("Chargement de BLIP en fp16 pour le test programmatique...")
test_model, test_processor = load_model_and_processor("blip", "fp16")

# 1. Génération
generated_cap = generate_image_caption(test_model, test_processor, sample_img, "blip")
print(f"\nLégende générée par BLIP : '{generated_cap}'")

# 2. Jugement de deux légendes
caption_good = "A happy dog looking closely at the camera outdoors"
caption_bad = "A large commercial airplane flying through stormy clouds"

judge_good = judge_single_caption(test_model, test_processor, sample_img, caption_good)
judge_bad = judge_single_caption(test_model, test_processor, sample_img, caption_bad)
comp = compare_captions(test_model, test_processor, sample_img, caption_good, caption_bad)

print(f"\nÉvaluation Légende 1 ('{caption_good}') :")
print(f"  -> Loss : {judge_good['loss']} | Score : {judge_good['score']} | Verdict : {judge_good['verdict']}")

print(f"\nÉvaluation Légende 2 ('{caption_bad}') :")
print(f"  -> Loss : {judge_bad['loss']} | Score : {judge_bad['score']} | Verdict : {judge_bad['verdict']}")

print(f"\nComparaison A/B : Choix du modèle = {comp['winner']} (Écart de perte : {comp['delta']})")

# Visualisation
plt.figure(figsize=(6, 4))
plt.imshow(sample_img)
plt.axis("off")
plt.title(f"Génération : {generated_cap}", fontsize=11)
plt.show()

# Libérer la mémoire
del test_model, test_processor
clear_gpu()

## 3. Interface Graphique Interactive Gradio

L'interface ci-dessous permet d'interagir facilement avec les modèles :
- Sélecteur de modèle (**BLIP** ou **GIT**)
- Sélecteur de niveau de quantification (**fp16**, **int8**, **int4**)
- Import d'image par glisser-déposer
- Trois modes d'utilisation : Légendage, Jugement d'une légende, Comparaison de 2 légendes.

Le paramètre `share=True` permet d'obtenir une URL publique pour partager la démo.

In [ ]:
import gradio as gr

STATE = {"model": None, "processor": None, "config": None}


def get_demo_model(model_name, precision):
    cfg = (model_name.lower(), precision.lower())
    if STATE["model"] is not None and STATE["config"] == cfg:
        return STATE["model"], STATE["processor"], "Modèle en cache (prêt)"
    if STATE["model"] is not None:
        del STATE["model"]
        del STATE["processor"]
        clear_gpu()
    t0 = time.perf_counter()
    m, p = load_model_and_processor(model_name, precision)
    STATE["model"], STATE["processor"], STATE["config"] = m, p, cfg
    return m, p, f"Modèle chargé en {time.perf_counter() - t0:.2f} s"


def ui_generate(img, m_name, prec):
    if img is None:
        return "Veuillez charger une image.", ""
    m, p, status = get_demo_model(m_name, prec)
    t0 = time.perf_counter()
    cap = generate_image_caption(m, p, img, m_name)
    info = f"{status} | Inférence : {time.perf_counter() - t0:.2f} s | {m_name.upper()} ({prec})"
    return cap, info


def ui_judge(img, caption, m_name, prec):
    if img is None or not caption or not caption.strip():
        return "Renseignez une image et une légende.", "", "", ""
    m, p, status = get_demo_model(m_name, prec)
    t0 = time.perf_counter()
    res = judge_single_caption(m, p, img, caption.strip())
    info = f"{status} | Calculé en {time.perf_counter() - t0:.2f} s"
    return f"{res['loss']:.4f}", f"{res['score']:.4f} (PPL: {res['perplexity']})", res["verdict"], info


def ui_compare(img, cap_a, cap_b, m_name, prec):
    if img is None or not cap_a or not cap_b:
        return "Renseignez une image et deux légendes.", "", "", ""
    m, p, status = get_demo_model(m_name, prec)
    t0 = time.perf_counter()
    comp = compare_captions(m, p, img, cap_a.strip(), cap_b.strip())
    res_a = f"Loss : {comp['loss_a']:.4f}"
    res_b = f"Loss : {comp['loss_b']:.4f}"
    winner_str = f"Meilleure légende : {comp['winner']} (Écart : {comp['delta']:.4f})"
    info = f"{status} | Comparé en {time.perf_counter() - t0:.2f} s"
    return res_a, res_b, winner_str, info


with gr.Blocks(title="VLMs Légers - Démonstration", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Démonstration Interactive : BLIP & GIT")
    gr.Markdown("Évaluez et comparez les modèles vision-langage légers en précision normale (fp16) ou quantifiée (int8, int4).")

    with gr.Row():
        with gr.Column(scale=1):
            model_opt = gr.Radio(["blip", "git"], value="blip", label="Modèle VLM")
            prec_opt = gr.Dropdown(
                ["fp16", "int8", "int4"] if DEVICE == "cuda" else ["fp32"],
                value="fp16" if DEVICE == "cuda" else "fp32",
                label="Quantification",
            )
            image_input = gr.Image(type="pil", label="Image d'entrée")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("1. Légendage (Captioning)"):
                    btn_gen = gr.Button("Générer la légende", variant="primary")
                    out_gen = gr.Textbox(label="Légende prédite")
                    out_gen_meta = gr.Markdown()
                    btn_gen.click(ui_generate, [image_input, model_opt, prec_opt], [out_gen, out_gen_meta])

                with gr.TabItem("2. Jugement Image-Légende"):
                    txt_caption = gr.Textbox(label="Légende candidate à évaluer", placeholder="Entrez une description...")
                    btn_eval = gr.Button("Évaluer l'alignement", variant="primary")
                    with gr.Row():
                        out_loss = gr.Textbox(label="Perte (Loss)")
                        out_score = gr.Textbox(label="Score de cohérence (-Loss)")
                    out_verdict = gr.Textbox(label="Verdict d'alignement")
                    out_eval_meta = gr.Markdown()
                    btn_eval.click(ui_judge, [image_input, txt_caption, model_opt, prec_opt], [out_loss, out_score, out_verdict, out_eval_meta])

                with gr.TabItem("3. Comparaison A/B de légendes"):
                    cap_a = gr.Textbox(label="Légende A (ex: correcte)")
                    cap_b = gr.Textbox(label="Légende B (ex: perturbée)")
                    btn_cmp = gr.Button("Comparer les deux légendes", variant="primary")
                    with gr.Row():
                        out_a = gr.Textbox(label="Score Légende A")
                        out_b = gr.Textbox(label="Score Légende B")
                    out_winner = gr.Textbox(label="Décision du VLM")
                    out_cmp_meta = gr.Markdown()
                    btn_cmp.click(ui_compare, [image_input, cap_a, cap_b, model_opt, prec_opt], [out_a, out_b, out_winner, out_cmp_meta])

demo.launch(share=True, debug=False)

## 4. Synthèse et comparaison avec l'évaluation

Cette démonstration concrétise les résultats obtenus dans le **notebook 04** :
- **BLIP** se distingue par une formulation descriptive naturelle et une excellente discrimination entre paires vraies et fausses.
- **GIT** offre une alternative compacte et rapide.
- **Quantification 8-bit et 4-bit** : permet de réduire considérablement l'empreinte mémoire VRAM (divisée par ~2 en int8 et ~4 en int4) tout en préservant des performances d'alignement quasi-identiques à la version fp16.

In [ ]:
# Affichage des métriques issues du dossier results si disponible
summary_file = RESULTS_DIR / "model_judgement_summary.csv"
if summary_file.exists():
    import pandas as pd
    df_summary = pd.read_csv(summary_file)
    print("Métriques des modèles enregistrées lors de l'évaluation :")
    display(df_summary)
else:
    print("Fichier model_judgement_summary.csv introuvable sur Drive.")
    print("Exécutez le notebook 04_evaluation.ipynb pour générer les mesures complètes.")